# Instacart: limpieza y estandarización de datos



**Evidencia:** GA1-220501094-01-AA1-EV06  

**Lección 2:** Limpieza y estandarización  



Este notebook desarrolla un pipeline reproducible de limpieza con Pandas, genera un informe de cambios y resuelve siete ejercicios prácticos. Se conservan los datos originales en objetos separados y no se imputan los valores ausentes de `days_since_prior_order`, porque son esperables para el primer pedido de cada usuario.



> Los CSV incluidos en este repositorio son datos pequeños de ejemplo. Las conclusiones y conteos de esta ejecución no representan el conjunto completo de Instacart.

## 1. Preparar el entorno y cargar los datos



El bloque localiza la raíz del proyecto tanto si Jupyter se inicia desde `instacart-analisis/` como desde `instacart-analisis/notebooks/`. Los cinco CSV usan punto y coma como separador.

In [52]:
from pathlib import Path



import pandas as pd

from IPython.display import display



EXPECTED_FILES = (

    "instacart_orders.csv",

    "order_products.csv",

    "products.csv",

    "aisles.csv",

    "departments.csv",

)



candidate_roots = (Path.cwd(), Path.cwd().parent)

PROJECT_ROOT = next(

    (

        candidate.resolve()

        for candidate in candidate_roots

        if all((candidate / "data" / filename).is_file() for filename in EXPECTED_FILES)

    ),

    None,

)



if PROJECT_ROOT is None:

    raise FileNotFoundError(

        "No se encontraron los cinco CSV en data/. Inicia Jupyter desde "

        "instacart-analisis/ o desde instacart-analisis/notebooks/."

    )



DATA_DIR = PROJECT_ROOT / "data"

print(f"Raíz del proyecto: {PROJECT_ROOT}")

print(f"Directorio de datos: {DATA_DIR}")

Raíz del proyecto: D:\TRABAJOS SENA\CIENCIAS DE DATOS\FUNDAMENTOS EN PANDAS\instacart-analisis
Directorio de datos: D:\TRABAJOS SENA\CIENCIAS DE DATOS\FUNDAMENTOS EN PANDAS\instacart-analisis\data


In [53]:
raw_data = {

    Path(filename).stem: pd.read_csv(DATA_DIR / filename, sep=";")

    for filename in EXPECTED_FILES

}



orders_raw = raw_data["instacart_orders"]

order_products_raw = raw_data["order_products"]

products_raw = raw_data["products"]

aisles_raw = raw_data["aisles"]

departments_raw = raw_data["departments"]



print("Archivos cargados:", ", ".join(raw_data))

Archivos cargados: instacart_orders, order_products, products, aisles, departments


## 2. Diagnóstico inicial



Antes de modificar los datos, se revisan dimensiones, tipos, valores ausentes y duplicados. Esta línea base permite explicar qué cambió en el pipeline.

In [54]:
initial_structure = pd.DataFrame(

    [

        {

            "tabla": name,

            "filas": len(frame),

            "columnas": frame.shape[1],

            "valores_ausentes": int(frame.isna().sum().sum()),

            "filas_duplicadas_exactas": int(frame.duplicated().sum()),

        }

        for name, frame in raw_data.items()

    ]

)

display(initial_structure)



for name, frame in raw_data.items():

    print(f"\n{name}: {frame.shape}")

    display(frame.dtypes.rename("tipo").to_frame())

,tabla,filas,columnas,valores_ausentes,filas_duplicadas_exactas
0,instacart_orders,4,7,3,0
1,order_products,4,4,0,0
2,products,3,4,0,0
3,aisles,3,2,0,0
4,departments,2,2,0,0



instacart_orders: (4, 7)


,tipo
order_id,int64
user_id,int64
eval_set,str
order_number,int64
order_dow,int64
order_hour_of_day,int64
days_since_prior_order,float64



order_products: (4, 4)


,tipo
order_id,int64
product_id,int64
add_to_cart_order,int64
reordered,int64



products: (3, 4)


,tipo
product_id,int64
product_name,str
aisle_id,int64
department_id,int64



aisles: (3, 2)


,tipo
aisle_id,int64
aisle,str



departments: (2, 2)


,tipo
department_id,int64
department,str


## 3. Laboratorio guiado 2: pipeline de limpieza con informe de cambios

El laboratorio recarga los archivos originales, registra una fotografia de cada paso (filas, nulos y memoria), aplica funciones pequenas y encadena la limpieza con DataFrame.pipe. Los datos crudos se mantienen intactos para poder comparar los resultados y repetir el proceso.

In [55]:
registro = []


def anotar(df, tabla, paso):
    registro.append(
        {
            "tabla": tabla,
            "paso": paso,
            "filas": len(df),
            "nulos": int(df.isna().sum().sum()),
            "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
        }
    )
    return df


def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)


def tipos_orders(df):
    return df.astype(
        {
            "order_id": "int32",
            "user_id": "int32",
            "order_number": "int16",
            "order_dow": "int8",
            "order_hour_of_day": "int8",
            "days_since_prior_order": "Int16",
        }
    )


def limpiar_nombres(df):
    df = df.copy()
    df["product_name"] = (
        df["product_name"]
        .fillna("Unknown")
        .str.replace("\xa0", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.title()
    )
    return df


def tipos_order_prods(df):
    return df.astype(
        {
            "order_id": "int32",
            "product_id": "int32",
            "add_to_cart_order": "Int16",
            "reordered": "int8",
        }
    )

In [56]:
orders_crudo = pd.read_csv(DATA_DIR / "instacart_orders.csv", sep=";")
products_crudo = pd.read_csv(DATA_DIR / "products.csv", sep=";")
op_crudo = pd.read_csv(DATA_DIR / "order_products.csv", sep=";")

print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(4, 7) (3, 4) (4, 4)


In [57]:
orders_limpio = (
    orders_crudo.pipe(anotar, "orders", "0 crudo")
    .pipe(quitar_duplicados)
    .pipe(anotar, "orders", "1 sin duplicados")
    .pipe(tipos_orders)
    .pipe(anotar, "orders", "2 tipos")
)

products_limpio = (
    products_crudo.pipe(anotar, "products", "0 crudo")
    .pipe(limpiar_nombres)
    .pipe(anotar, "products", "1 nombres")
)

op_limpio = (
    op_crudo.pipe(anotar, "order_prods", "0 crudo")
    .pipe(tipos_order_prods)
    .pipe(anotar, "order_prods", "1 tipos")
)

orders_clean = orders_limpio
products_clean = products_limpio
order_products_clean = op_limpio
order_prods = order_products_clean
aisles_clean = aisles_raw.copy()
departments_clean = departments_raw.copy()

In [58]:
cleaning_report = pd.DataFrame(registro)
display(cleaning_report)

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,4,3,0.0
1,orders,1 sin duplicados,4,3,0.0
2,orders,2 tipos,4,3,0.0
3,products,0 crudo,3,0,0.0
4,products,1 nombres,3,0,0.0
5,order_prods,0 crudo,4,0,0.0
6,order_prods,1 tipos,4,0,0.0


### Lectura del informe

El informe conserva el estado de cada tabla en cada paso, asi se puede ver si la limpieza cambio filas, nulos o memoria. Los nombres de producto que faltan se sustituyen por "Unknown" para conservar las filas de articulos; los nulos de add_to_cart_order se investigan en el Reto 2, no se imputan sin conocer su causa. Nombres de producto iguales no implican el mismo producto: distintos product_id se conservan para no perder referencias de pedidos.

In [59]:
def validar(orders, products, order_prods):
    assert orders["order_id"].is_unique, "order_id repetido"
    assert orders["order_hour_of_day"].between(0, 23).all(), "hora fuera de rango"
    assert products["product_name"].notna().all(), "productos sin nombre"
    assert not products["product_name"].str.contains(
        "\xa0", regex=False
    ).any(), "espacios invisibles"
    assert order_prods["reordered"].isin([0, 1]).all(), "reordered con valores raros"
    return "Validacion superada"


validation_result = validar(orders_clean, products_clean, order_products_clean)
assert orders_crudo.equals(orders_raw), "La tabla cruda de pedidos fue modificada"
assert products_crudo.equals(products_raw), "La tabla cruda de productos fue modificada"
print(validation_result)
print("Las copias de datos crudos permanecen intactas.")

Validacion superada
Las copias de datos crudos permanecen intactas.


## 4. Reto 2: siete ejercicios de limpieza



Los ejercicios siguientes practican detección, estandarización y validación usando los cinco DataFrames limpiados. Las soluciones muestran resultados verificables y conservan la trazabilidad de los cambios.

### Ejercicio 1. Revisar duplicados y valores de reordered

Comprueba si order_prods tiene filas duplicadas exactas, si se repite algun par order_id + product_id y si reordered solo contiene 0 y 1.

In [60]:
exact_duplicates = int(order_prods.duplicated().sum())
duplicate_pairs = order_prods.duplicated(
    subset=["order_id", "product_id"], keep=False
)
duplicate_pair_count = order_prods.loc[
    duplicate_pairs, ["order_id", "product_id"]
].drop_duplicates().shape[0]
reordered_values = sorted(order_prods["reordered"].dropna().unique().tolist())
reordered_is_binary = order_prods["reordered"].isin([0, 1]).all()

display(
    pd.DataFrame(
        {
            "resultado": [
                exact_duplicates,
                duplicate_pair_count,
                reordered_values,
                reordered_is_binary,
            ]
        },
        index=[
            "filas duplicadas exactas",
            "pares order_id-product_id repetidos",
            "valores de reordered",
            "reordered contiene solo 0 y 1",
        ],
    )
)

,resultado
filas duplicadas exactas,0
pares order_id-product_id repetidos,0
valores de reordered,"[0, 1]"
reordered contiene solo 0 y 1,True


**Interpretacion:** una repeticion de order_id + product_id senalaria el mismo producto repetido dentro de un pedido; valores de reordered distintos de 0 y 1 requieren investigar antes de analizarlos.

### Ejercicio 2. Revisar nombres de pasillos y departamentos

Comprueba si aisles y departments tienen nombres duplicados o con espacios al inicio o al final. Decide si necesitan limpieza de texto.

In [61]:
text_quality_rows = []
for table_name, frame, name_column in [
    ("aisles", aisles_raw, "aisle"),
    ("departments", departments_raw, "department"),
]:
    names = frame[name_column].astype("string")
    text_quality_rows.append(
        {
            "tabla": table_name,
            "nombres_duplicados": int(names.dropna().duplicated().sum()),
            "nombres_con_espacios_laterales": int(
                (names.notna() & names.ne(names.str.strip())).sum()
            ),
        }
    )

text_quality_report = pd.DataFrame(text_quality_rows)
display(text_quality_report)

,tabla,nombres_duplicados,nombres_con_espacios_laterales
0,aisles,1,0
1,departments,0,0


**Interpretacion:** si los conteos de espacios laterales son cero, ese defecto no requiere correccion en estas tablas. Los nombres duplicados se evaluan segun su funcion: varios ids pueden compartir una etiqueta.

### Ejercicio 3. Investigar los nulos de add_to_cart_order

El material identifica 836 nulos. Determina en cuantos pedidos ocurren y cuantos productos tiene, como minimo, cada pedido afectado. No elimines esas filas sin una causa comprobada.

In [62]:
missing_cart_rows = order_prods.loc[order_prods["add_to_cart_order"].isna()]
missing_cart_order_ids = missing_cart_rows["order_id"].drop_duplicates()
products_per_order = order_prods.groupby("order_id").size()
missing_cart_order_sizes = (
    products_per_order.loc[products_per_order.index.isin(missing_cart_order_ids)]
    .sort_values()
    .rename("productos_en_pedido")
    .rename_axis("order_id")
    .reset_index()
)

print(f"Filas con add_to_cart_order nulo: {len(missing_cart_rows):,}")
print(f"Pedidos afectados: {len(missing_cart_order_ids):,}")
if missing_cart_order_sizes.empty:
    print("Esta muestra no contiene pedidos con posicion de carrito nula.")
else:
    print(
        "Minimo de productos por pedido afectado:",
        int(missing_cart_order_sizes["productos_en_pedido"].min()),
    )
    display(missing_cart_order_sizes)

Filas con add_to_cart_order nulo: 0
Pedidos afectados: 0
Esta muestra no contiene pedidos con posicion de carrito nula.


**Interpretacion:** la fila del articulo contiene un pedido y un producto aunque falte la posicion en el carrito. Conservamos esos registros y describimos cuantos articulos hay en cada pedido afectado; la muestra reducida puede no incluir los 836 casos del conjunto completo.

### Ejercicio 4. Contar nombres cambiados por la estandarizacion

Carga de nuevo products.csv en products_crudo y cuenta cuantos nombres no nulos cambiaron despues de estandarizarlos.

In [67]:
products_crudo_reto = pd.read_csv(DATA_DIR / "products.csv", sep=";")
products_estandarizados_reto = limpiar_nombres(products_crudo_reto)
original_names = products_crudo_reto["product_name"]
standardized_names = products_estandarizados_reto["product_name"]
changed_non_null_names = original_names.notna() & original_names.ne(standardized_names)
changed_name_count = int(changed_non_null_names.sum())

print(f"Nombres no nulos que cambiaron: {changed_name_count:,}")
display(
    pd.DataFrame(
        {
            "original": original_names.loc[changed_non_null_names],
            "estandarizado": standardized_names.loc[changed_non_null_names],
        }
    ).head(10)
)

Nombres no nulos que cambiaron: 0


,original,estandarizado


**Interpretacion:** el conteo incluye solo nombres que existian antes de limpiar; los nulos originales, aunque se convierten en "Unknown", no se cuentan como nombres cambiados.

### Ejercicio 5. Comparar memoria de dia como texto y categoria ordenada

Crea una Series con el nombre del dia de cada pedido, conviertela a una categoria ordenada de domingo a sabado y compara la memoria utilizada.

In [68]:
weekday_names = ["domingo", "lunes", "martes", "mi\u00e9rcoles", "jueves", "viernes", "s\u00e1bado"]
weekday_text = orders_clean["order_dow"].map(
    dict(enumerate(weekday_names))
).astype("string")
weekday_category = pd.Series(
    pd.Categorical(weekday_text, categories=weekday_names, ordered=True),
    index=orders_clean.index,
    name="dia_categoria",
)

weekday_memory = pd.DataFrame(
    {
        "memoria_bytes": [
            int(weekday_text.memory_usage(deep=True)),
            int(weekday_category.memory_usage(deep=True)),
        ]
    },
    index=["texto", "categoria ordenada"],
)
display(weekday_memory)
display(pd.DataFrame({"texto": weekday_text, "categoria": weekday_category}).head())

,memoria_bytes
texto,353
categoria ordenada,857


,texto,categoria
0,domingo,domingo
1,lunes,lunes
2,domingo,domingo
3,martes,martes


**Interpretacion:** compara las dos cifras de memoria. Las categorias permiten ademas fijar el orden natural de los dias, independientemente de su orden alfabetico.

### Ejercicio 6. Revisar filas de productos Unknown

Cuantas filas de order_prods corresponden a productos cuyo product_name es "Unknown" y en cuantos pedidos aparecen? Decide si se deben eliminar.

In [65]:
order_prods_with_names = order_prods.merge(
    products_clean[["product_id", "product_name"]],
    on="product_id",
    how="left",
    validate="many_to_one",
)
unknown_items = order_prods_with_names.loc[
    order_prods_with_names["product_name"].eq("Unknown")
]
unknown_item_rows = len(unknown_items)
unknown_item_orders = int(unknown_items["order_id"].nunique())

print(f"Filas de productos Unknown: {unknown_item_rows:,}")
print(f"Pedidos con productos Unknown: {unknown_item_orders:,}")

Filas de productos Unknown: 0
Pedidos con productos Unknown: 0


**Interpretacion:** no se eliminan: son articulos que si aparecen en pedidos, aunque falte su nombre de catalogo. Quitarlos sesgaria los conteos y perderia referencias validas de compra.

### Ejercicio 7. Reemplazar missing en una copia de departments

En una copia de departments, reemplaza "missing" por pd.NA. Explica por que no conviene hacerlo en la tabla original antes de agrupar.

In [66]:
departments_for_analysis = departments_raw.copy()
departments_for_analysis["department"] = departments_for_analysis["department"].replace(
    "missing", pd.NA
)
display(departments_for_analysis)
print("La tabla original conserva sus valores:", departments_raw["department"].tolist())

,department_id,department
0,1,produce
1,2,dairy


La tabla original conserva sus valores: ['produce', 'dairy']


## 5. Conclusiones de la Leccion 2

- El pipeline reutilizable registra filas, nulos y memoria antes y despues de cada paso.
- Los tipos numericos quedan ajustados a su rango y las posiciones ausentes del carrito se conservan para investigar su causa.
- Los nombres de producto se estandarizan sin eliminar articulos ni confundir productos con ids distintos.
- Los siete ejercicios revisan duplicados, reglas de reordered, calidad de nombres, nulos en el carrito, estandarizacion, memoria categorica y valores "missing".
- Los resultados de estos CSV de ejemplo no sustituyen los resultados del conjunto completo de Instacart.